In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, RobustScaler


In [3]:
train_df = pd.read_parquet("../DATA/proprocessed_data/train_df_preprocessed.parquet")
test_df = pd.read_parquet("../DATA/proprocessed_data/test_df_preprocessed.parquet")
model_features = pickle.load(open("../pickle_files/model_features.pkl", "rb"))

In [4]:
model_features.append("default")

In [5]:
model_features

['loan_purpose',
 'loan_type',
 'loan_tenure_months',
 'age',
 'residence_type',
 'number_of_open_accounts',
 'credit_utilization_ratio',
 'loan_to_income',
 'delinquency_ratio',
 'avg_dpd_per_delinquency',
 'default']

In [6]:
train_df_v1 = train_df[model_features]
test_df_v1 = test_df[model_features]

In [7]:
train_df.shape , test_df.shape

((33500, 36), (16500, 36))

In [8]:
train_df_v1.shape,test_df_v1.shape

((33500, 11), (16500, 11))

In [9]:
train_df_v1.dtypes

loan_purpose                 object
loan_type                    object
loan_tenure_months            int64
age                           int64
residence_type               object
number_of_open_accounts       int64
credit_utilization_ratio      int64
loan_to_income              float64
delinquency_ratio           float64
avg_dpd_per_delinquency     float64
default                       int32
dtype: object

In [10]:
X_train = train_df_v1.drop("default",axis=1)
y_train = train_df_v1["default"]

X_test = test_df_v1.drop("default",axis=1)
y_test = test_df_v1["default"]

In [11]:
#ENCODING WITH OneHotEncoder

cat_cols = X_train.select_dtypes(include="object").columns
encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
encoder.fit(X_train[cat_cols])
pickle.dump(encoder,open("../pickle_files/encoder.pkl", "wb"))


In [12]:
encoder = pickle.load(open("../pickle_files/encoder.pkl", "rb"))

X_train_cat_encoded = encoder.transform(X_train[cat_cols])
X_test_cat_encoded = encoder.transform(X_test[cat_cols])

In [13]:
encoded_train = pd.DataFrame(X_train_cat_encoded,columns=encoder.get_feature_names_out(), index= X_train.index)
encoded_test = pd.DataFrame(X_test_cat_encoded,columns=encoder.get_feature_names_out(),index=X_test.index)

In [14]:
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns

In [15]:
scaler = RobustScaler()
scaler.fit(X_train[num_cols])
pickle.dump(scaler,open("../pickle_files/scaler.pkl", "wb"))


In [16]:
scaler = pickle.load(open("../pickle_files/scaler.pkl", "rb"))

X_train_num_scaled = scaler.transform(X_train[num_cols])
X_test_num_scaled = scaler.transform(X_test[num_cols])

In [17]:
sc_train = pd.DataFrame(X_train_num_scaled,columns=num_cols)
sc_test = pd.DataFrame(X_test_num_scaled,columns=num_cols)



In [18]:
X_train_model = pd.concat([encoded_train,sc_train],axis=1)
X_test_model = pd.concat([encoded_test,sc_test],axis=1)

In [19]:
X_train_model

,loan_purpose_Auto,loan_purpose_Education,loan_purpose_Home,loan_purpose_Personal,loan_type_Secured,loan_type_Unsecured,residence_type_Mortgage,residence_type_Owned,residence_type_Rented,loan_tenure_months,age,number_of_open_accounts,credit_utilization_ratio,loan_to_income,delinquency_ratio,avg_dpd_per_delinquency
0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,-0.631579,-0.230769,-0.5,0.775510,-0.420118,0.403101,0.280702
1,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.263158,0.307692,-1.0,-0.489796,-0.065089,6.031008,0.403509
2,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,-0.631579,0.230769,-1.0,-0.387755,-0.307692,1.519380,0.298246
3,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,-0.210526,0.615385,0.0,0.816327,-0.325444,-0.286822,-0.754386
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.263158,-0.384615,-1.0,-0.591837,1.278107,2.294574,-0.578947
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33495,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.526316,-0.307692,0.5,-0.102041,0.970414,-0.286822,-0.754386
33496,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.894737,1.307692,-0.5,-0.591837,-0.017751,-0.286822,-0.754386
33497,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,-0.631579,2.384615,-0.5,-0.530612,-0.349112,-0.286822,-0.754386
33498,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.421053,0.000000,-1.0,0.102041,-0.029586,1.798450,0.140351


In [20]:
train_df_final = pd.concat([X_train_model, y_train.to_frame()],axis=1)
test_df_final = pd.concat([X_test_model, y_test.to_frame()],axis=1)

In [21]:
test_df_final.shape

(16500, 17)

In [22]:
#by uding columntransformer and pipeline 

In [23]:
import pandas as pd
import pickle

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,RobustScaler


train_df = pd.read_parquet("../DATA/proprocessed_data/train_df_preprocessed.parquet")
test_df = pd.read_parquet("../DATA/proprocessed_data/test_df_preprocessed.parquet")
model_features = pickle.load(open("../pickle_files/model_features.pkl", "rb"))
model_features.append("default")



In [24]:
train_df_v1 = train_df[model_features]
test_df_v1 = test_df[model_features]


In [25]:
X_train = train_df_v1.drop("default", axis=1)
y_train = train_df_v1["default"]

X_test = test_df_v1.drop("default", axis=1)
y_test = test_df_v1["default"]


In [26]:
cat_cols = X_train.select_dtypes(include="object").columns
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns


In [27]:
preprocessor = ColumnTransformer(transformers=[
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols),
    ("num",RobustScaler(),num_cols)])


In [28]:
pipeline = Pipeline([("preprocessor", preprocessor)])


In [29]:
pipeline.fit(X_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse ma

In [30]:
pickle.dump(pipeline,open("../pickle_files/pipeline.pkl", "wb"))


In [31]:
pipeline = pickle.load(open("../pickle_files/pipeline.pkl", "rb"))
X_train_model = pipeline.transform(X_train)
X_test_model = pipeline.transform(X_test)


In [32]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

In [33]:
X_train_model = pd.DataFrame(
    X_train_model,
    columns=feature_names,
    index=X_train.index
)

X_test_model = pd.DataFrame(
    X_test_model,
    columns=feature_names,
    index=X_test.index
)


In [34]:
train_df_final = pd.concat(
    [X_train_model, y_train.to_frame()],
    axis=1
)

test_df_final = pd.concat(
    [X_test_model, y_test.to_frame()],
    axis=1
)


In [35]:

pickle.dump(num_cols,open("../pickle_files/num_cols.pkl", "wb"))
pickle.dump(cat_cols,open("../pickle_files/cat_cols.pkl", "wb"))

In [36]:
train_df_final.to_parquet("../DATA/model/train_df_for_model.parquet")
test_df_final.to_parquet("../DATA/model/test_df_for_model.parquet")